Este notebook demonstra o treinamento de redes neurais para classificação binária, comparando um modelo sem regularização, outro com Dropout e um terceiro com callbacks de treinamento.


### **Importações**

#### Bibliotecas e funções utilizadas


In [ ]:
# Numpy
import numpy as np

# MatPlot Lib
import matplotlib.pyplot as plt

# Scikit Learn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Tensor Flow

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras import callbacks

#### Criação do conjunto de dados e separação entre treino e teste


In [ ]:
# Gera exemplos sintéticos de duas classes em formato de luas.
X_bruto, y = make_moons(n_samples=250,
                        noise=0.35,
                        random_state=42)

# Separa os dados para treinamento e avaliação.
X_train, X_test, y_train, y_test = train_test_split(X_bruto,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y)

#### Padronização das variáveis


In [ ]:
# Cria o padronizador das características.
scaler = StandardScaler()

# Aprende a escala com os dados de treino e aplica a transformação.
X_train_scaled = scaler.fit_transform(X_train)

# Aplica ao teste a mesma escala aprendida no treino.
X_test_scaled = scaler.transform(X_test)

#### Visualização das classes


In [ ]:
# Prepara a área para a visualização gráfica.
plt.figure(figsize=(8, 4.5))

# Plota os pontos para observar a distribuição das classes.
plt.scatter(X_train_scaled[y_train == 0, 0], X_train_scaled[y_train == 0,1], color = '#e11d48', label = 'Classe 0', alpha = 0.6, s = 35)

# Plota os pontos para observar a distribuição das classes.
plt.scatter(X_train_scaled[y_train == 1, 0], X_train_scaled[y_train == 1,1], color = '#0284c7', label = 'Classe 1', alpha = 0.6, s = 35)
plt.title('Dataset')
plt.xlabel('Atributo 1 (padronizado)')
plt.ylabel('Atributo 2 (padronizado)')
plt.legend(frameon=True, facecolor='white')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Modelo Over

Primeiro, treinamos uma rede neural sem Dropout para observar o comportamento das métricas de treinamento e validação.


#### Arquitetura do modelo sem Dropout


In [ ]:
# Define a semente aleatória para favorecer a reprodutibilidade.
tf.keras.utils.set_random_seed(42)

# Define as camadas da rede neural, na ordem em que serão executadas.
modelo_over = Sequential([
    Input(shape=(2,)),
    Dense(128, activation='relu', name='oculta_1'),
    Dense(64, activation='relu', name='oculta_2'),
    Dense(32, activation='relu', name='oculta_3'),
    Dense(1, activation='sigmoid', name='saida')
])

#### Configuração do treinamento


In [ ]:
# Escolhe o otimizador, a função de perda e a métrica de avaliação.
modelo_over.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Mostra um resumo das camadas e dos parâmetros do modelo.
modelo_over.summary()

#### Treinamento do modelo sem Dropout


In [ ]:
# Treina a rede e registra as métricas ao longo das épocas.
hist_over = modelo_over.fit(
    X_train_scaled,
    y_train,
    epochs=120,
    batch_size=32,
    validation_split=0.25,
    verbose=1,
)

print('\nTreinamento do Modelo Over: \033[1;32mConcluído com sucesso\033[0m')

### Modelo Dropout

Nesta etapa, adicionamos Dropout para reduzir a dependência de neurônios específicos durante o treinamento.


#### Arquitetura do modelo com Dropout


In [ ]:
# Define a semente aleatória para favorecer a reprodutibilidade.
tf.keras.utils.set_random_seed(42)

# Define as camadas da rede neural, na ordem em que serão executadas.
modelo_dropout = Sequential([
    Input(shape=(2,)),
    Dense(128, activation='relu', name='oculta_1'),
    Dropout(0.3, name='dropout_1'),
    Dense(64, activation='relu', name='oculta_2'),
    Dropout(0.3, name='dropout_2'),
    Dense(32, activation='relu', name='oculta_3'),
    Dropout(0.3, name='dropout_3'),
    Dense(1, activation='sigmoid', name='saida')
])

#### Configuração do modelo com Dropout


In [ ]:
# Escolhe o otimizador, a função de perda e a métrica de avaliação.
modelo_dropout.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Mostra um resumo das camadas e dos parâmetros do modelo.
modelo_dropout.summary()

#### Treinamento do modelo com Dropout


In [ ]:
# Treina a rede e registra as métricas ao longo das épocas.
hist_dropout = modelo_dropout.fit(
    X_train_scaled,
    y_train,
    epochs=120,
    batch_size=32,
    validation_split=0.25,
    verbose=1,
)

print('\nTreinamento do Modelo Dropout: \033[1;32mConcluído com sucesso\033[0m')

#### Comparação visual dos históricos


In [ ]:
# Prepara a área para a visualização gráfica.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico do Modelo SEM DROPOUT

# Desenha a evolução das métricas registradas.
ax1.plot(hist_over.history['loss'],
         label='Treino (Loss)',
         color='#0284c7',
         linewidth=2)
# Desenha a evolução das métricas registradas.
ax1.plot(hist_over.history['val_loss'],
         label='Validação (Val. Loss)',
         color='#e11d48',
         linestyle='--',
         linewidth=2.2)
ax1.set_title('Modelo SEM DROPOUT (Overfitting Severo)')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('Perda (Loss)')
ax1.set_ylim(0, 1.4)
ax1.legend(frameon=True, facecolor='white')
ax1.grid(True, linestyle='--', alpha=0.5)

# Gráfico do Modelo COM DROPOUT

# Desenha a evolução das métricas registradas.
ax2.plot(hist_dropout.history['loss'],
         label='Treino (Loss)',
         color='#0284c7',
         linewidth=2)
# Desenha a evolução das métricas registradas.
ax2.plot(hist_dropout.history['val_loss'],
         label='Validação (Val. Loss)',
         color='#e11d48',
         linestyle='--',
         linewidth=2.2)
ax2.set_title('Modelo COM DROPOUT')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Perda (Loss)')
ax2.set_ylim(0, 1.4)
ax2.legend(frameon=True, facecolor='white')
ax2.grid(True, linestyle='--', alpha=0.5)

### Modelo Aut

O último modelo utiliza Dropout e callbacks para acompanhar a validação e controlar automaticamente o treinamento.


#### Arquitetura do modelo com controle automático


In [ ]:
# Define a semente aleatória para favorecer a reprodutibilidade.
tf.keras.utils.set_random_seed(42)
# Define as camadas da rede neural, na ordem em que serão executadas.
modelo_aut = Sequential([
    Input(shape=(2,)),
    Dense(128, activation='relu', name='oculta_1'),
    Dropout(0.3, name='dropout_1'),
    Dense(64, activation='relu', name='oculta_2'),
    Dropout(0.3, name='dropout_2'),
    Dense(32, activation='relu', name='oculta_3'),
    Dropout(0.3, name='dropout_3'),
    Dense(1, activation='sigmoid', name='saida')
])

#### Compilação do modelo


In [ ]:
# Escolhe o otimizador, a função de perda e a métrica de avaliação.
modelo_aut.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Mostra um resumo das camadas e dos parâmetros do modelo.
modelo_aut.summary()

#### Callbacks para acompanhar e controlar o treinamento


In [ ]:
# Agrupa os controles que serão executados durante o treinamento.
lista_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.001,
        verbose=1
    )
]

#### Treinamento com callbacks


In [ ]:
# Treina a rede e registra as métricas ao longo das épocas.
hist_aut = modelo_aut.fit(
    X_train_scaled,
    y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.25,
    callbacks=lista_callbacks,
    verbose=1,
)

print('\nTreinamento do Modelo Aut: \033[1;32mConcluído com sucesso\033[0m')

#### Visualização do histórico final


In [ ]:
# Conta quantas épocas foram realmente executadas.
epocas_reais = len(hist_aut.history['loss'])

# Prepara a área para a visualização gráfica.
plt.figure(figsize=(9, 4.5))
# Desenha a evolução das métricas registradas.
plt.plot(hist_aut.history['loss'],
         label='Treino (Loss)',
         color='#0284c7',
         linewidth=2)
# Desenha a evolução das métricas registradas.
plt.plot(hist_aut.history['val_loss'],
         label='Validação (Val. Loss)',
         color='#10b981',
         linewidth=2.5)
melhor_epoca = np.argmin(hist_aut.history['val_loss'])
menor_val_loss = hist_aut.history['val_loss'][melhor_epoca]
plt.axvline(x=melhor_epoca,
            color='#e11d48',
            linestyle=':',
            linewidth=2,
            label=f'Melhor Época: {melhor_epoca + 1}')
plt.title('Keras com EarlyStopping e ReduceLROnPlateau')
plt.xlabel('Épocas')
plt.ylabel('Perda (Loss)')
plt.ylim(0, 1.4)
plt.legend(frameon=True, facecolor='white')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()